In [1]:
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
user = "hats0902"
repo = "computer_arch_VLA"

clone_url = f"https://{token}@github.com/{user}/{repo}.git"
!git clone {clone_url}
%cd {repo}
!git config user.name "{user}"
!git config user.email "hatsuho.clover@gmail.com

# # 作業後
# !git add notebooks/ figures/ results/
# !git commit -m "Add baseline benchmark"
# !git push

Cloning into 'computer_arch_VLA'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 1), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 2.02 MiB | 1.63 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/computer_arch_VLA
/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'


In [2]:
!pwd

/content/computer_arch_VLA


# 空間的局所性を確認する
行方向はメモリ上で隣り合っているので早くなる（単純に.sum()を読んだだけだとnumyp内部で工夫されているため違いが分かりにくい。そのためわざわざスライス処理をすることとした。）  
-->隣り合っていない値を飛び飛びに読むので、まとめて持ってきたデータが無駄になり、CPUの高速化の工夫も効きにくい

In [3]:
import numpy as np, time
n = 8000
a = np.random.rand(n, n)

t = time.perf_counter()
for i in range(n):
    a[i, :].sum()          # 1行を取り出す：連続した64KBを読む
print("行ごと:", time.perf_counter() - t)

t = time.perf_counter()
for j in range(n):
    a[:, j].sum()          # 1列を取り出す：64KBおきに1要素ずつ読む
print("列ごと:", time.perf_counter() - t)

行ごと: 0.09285408499999903
列ごと: 0.80185140399999


# T4環境の確認
--> `cuda.cuDeviceGetAttribute` で実行環境のスペックが確認できる.
--> L1は総合計96KBだが、shared memory合計らしい

In [4]:
from collections import Counter
import ctypes

In [5]:
cuda = ctypes.CDLL("libcuda.so")

def check(err, what):
    if err != 0:
        raise RuntimeError(f"{what} が失敗しました（エラーコード {err}）")

check(cuda.cuInit(0), "cuInit")
dev = ctypes.c_int()
check(cuda.cuDeviceGet(ctypes.byref(dev), 0), "cuDeviceGet")

def attr(code):
    """cuDeviceGetAttribute で、GPUの属性を1つ取得する"""
    v = ctypes.c_int()
    check(cuda.cuDeviceGetAttribute(ctypes.byref(v), code, dev), f"属性{code}の取得")
    return v.value

# 属性の番号は CUDA Driver API の CUdevice_attribute（cuda.h）で定義されている
CU_DEVICE_ATTRIBUTE = {
    "MULTIPROCESSOR_COUNT": 16,                    # SMの数
    "MEMORY_CLOCK_RATE": 36,                       # メモリクロック（kHz）
    "GLOBAL_MEMORY_BUS_WIDTH": 37,                 # メモリのバス幅（bit）
    "L2_CACHE_SIZE": 38,                           # L2キャッシュ（バイト）
    "COMPUTE_CAPABILITY_MAJOR": 75,
    "COMPUTE_CAPABILITY_MINOR": 76,
    "MAX_SHARED_MEMORY_PER_MULTIPROCESSOR": 81,    # shared memoryの上限（SMごと、バイト）
}

name_buf = ctypes.create_string_buffer(256)
check(cuda.cuDeviceGetName(name_buf, 256, dev), "cuDeviceGetName")
name = name_buf.value.decode()

cc = (attr(CU_DEVICE_ATTRIBUTE["COMPUTE_CAPABILITY_MAJOR"]),
      attr(CU_DEVICE_ATTRIBUTE["COMPUTE_CAPABILITY_MINOR"]))   # T4は 7.5
NUM_SM = attr(CU_DEVICE_ATTRIBUTE["MULTIPROCESSOR_COUNT"])
L2_BYTES = attr(CU_DEVICE_ATTRIBUTE["L2_CACHE_SIZE"])
SMEM_PER_SM = attr(CU_DEVICE_ATTRIBUTE["MAX_SHARED_MEMORY_PER_MULTIPROCESSOR"])

print(f"GPU: {name}（compute capability {cc[0]}.{cc[1]}）")
print(f"SMの数: {NUM_SM}")
print(f"L2キャッシュ: {L2_BYTES / 1024**2:.1f} MiB（全SMで共有）")
print(f"shared memoryの上限: SMごとに {SMEM_PER_SM / 1024:.0f} KiB")

GPU: Tesla T4（compute capability 7.5）
SMの数: 40
L2キャッシュ: 4.0 MiB（全SMで共有）
shared memoryの上限: SMごとに 64 KiB


# まずランダムな入力を使って推論
-->デフォルトのモデルのdatatypeはFP32

In [21]:
import ctypes
import torch
import torch.nn as nn
import pandas as pd

In [19]:
import torch, timm
device = "cuda" if torch.cuda.is_available() else "cpu"
model = timm.create_model("vit_base_patch16_224", pretrained=True).to(device).eval()

x = torch.randn(1, 3, 224, 224, device=device)
with torch.inference_mode():
    logits = model(x)
print(x.shape, "→", logits.shape)   # (1, 3, 224, 224) → (1, 1000)


torch.Size([1, 3, 224, 224]) → torch.Size([1, 1000])


In [9]:
for name, module in model.named_children(): # patchembed→transformer bloack→headなどという粒度でしか見れない（encoder blockの中身までは見れない）
    total_params = sum(p.numel() for p in module.parameters())
    # 学習可能パラメータ数
    trainable_params = sum(p.numel() for p in module.parameters() if p.requires_grad)

    print(f"{name:<20} | {total_params:>15,} | {trainable_params:>18,}")

    count, nbytes = Counter(), Counter()
    for p in module.parameters():
        count[p.dtype] += p.numel()
        nbytes[p.dtype] += p.numel() * p.element_size()
    for dt in count:
        print(dt, f"{count[dt]/1e6:.1f} M個", f"{nbytes[dt]/1e6:.0f} MB") # pos_drop などは重みを持たないため表示されない

patch_embed          |         590,592 |            590,592
torch.float32 0.6 M個 2 MB
pos_drop             |               0 |                  0
patch_drop           |               0 |                  0
norm_pre             |               0 |                  0
blocks               |      85,054,464 |         85,054,464
torch.float32 85.1 M個 340 MB
norm                 |           1,536 |              1,536
torch.float32 0.0 M個 0 MB
fc_norm              |               0 |                  0
head_drop            |               0 |                  0
head                 |         769,000 |            769,000
torch.float32 0.8 M個 3 MB


In [28]:
# 合計パラメーター数を数える
from collections import Counter

def count_params(model):
    """モデル全体のパラメータ数・学習可能なパラメータ数・dtypeごとのサイズを返す"""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    nbytes = Counter()
    for p in model.parameters():
        nbytes[p.dtype] += p.numel() * p.element_size()

    print(f"合計パラメータ数    : {total:,}（{total / 1e6:.2f} M）")
    print(f"学習可能なパラメータ: {trainable:,}")
    for dt, nb in nbytes.items():
        print(f"{dt}: {nb / 1e6:.1f} MB")
    return {"total": total, "trainable": trainable, "bytes_by_dtype": dict(nbytes)}

result = count_params(model)
result

合計パラメータ数    : 86,567,656（86.57 M）
学習可能なパラメータ: 86,567,656
torch.float32: 346.3 MB


{'total': 86567656,
 'trainable': 86567656,
 'bytes_by_dtype': {torch.float32: 346270624}}

In [10]:
L1_MAX_BY_CC = {
    (7, 5): 64 * 1024,    # Turing（T4）
}
L1_BYTES = L1_MAX_BY_CC.get(cc)
if L1_BYTES is None:
    L1_BYTES = SMEM_PER_SM   # 表にない世代は、shared memoryの上限で代用（目安）
    print(f"L1の最大量: 表に {cc} がないため shared memory の上限 {L1_BYTES / 1024:.0f} KiB で代用")
else:
    print(f"L1の最大量: SMごとに {L1_BYTES / 1024:.0f} KiB（Tuning Guideの値）")

L1の最大量: SMごとに 64 KiB（Tuning Guideの値）


In [12]:
total_all = sum(p.numel() for p in model.parameters())
print(f'max値 L1: {L1_BYTES}, L2: {L2_BYTES}')

def fits(nbytes):
    l1 = "○" if nbytes <= L1_BYTES else "✕"
    l2 = "○" if nbytes <= L2_BYTES else "✕"
    return f"L1:{l1}  L2:{l2}"

def show_row(name, params):
    """params: その部品のパラメータ（テンソル）のリスト"""
    total = sum(p.numel() for p in params)
    if total == 0:
        return   # drop や Identity など、重みを持たない部品は表示しない
    nbytes = Counter()
    for p in params:
        nbytes[p.dtype] += p.numel() * p.element_size()
    for dt, nb in nbytes.items():
        print(f"{name:<22} | {total:>12,} | {total / total_all * 100:5.1f}% | "
              f"{str(dt):<14} | {nb / 1e6:9.3f} MB | {fits(nb)}")

header = f"{'name':<22} | {'params':>12} | {'割合':>5} | {'dtype':<14} | {'容量':>12} | 収まるか"

# ① モデル直下の部品
print("===== モデル直下の部品 =====")
print(header)
for name, module in model.named_children():
    show_row(name, list(module.parameters()))

# cls_token と pos_embed は、部品（Module）ではなくモデルが直接持つ重みなので別に表示する
for name, p in model.named_parameters(recurse=False):
    show_row(name, [p])

# ② ブロック1個の中の部品（12ブロックとも同じ形）
print("\n===== ブロック1個の中の部品 =====")
print(header)
for name, module in model.blocks[0].named_modules():
    if name == "" or len(list(module.children())) > 0:
        continue   # ブロック自身と、attn・mlp などのまとまりは飛ばし、一番下の部品だけ表示
    show_row(name, list(module.parameters()))

max値 L1: 65536, L2: 4194304
===== モデル直下の部品 =====
name                   |       params |    割合 | dtype          |           容量 | 収まるか
patch_embed            |      590,592 |   0.7% | torch.float32  |     2.362 MB | L1:✕  L2:○
blocks                 |   85,054,464 |  98.3% | torch.float32  |   340.218 MB | L1:✕  L2:✕
norm                   |        1,536 |   0.0% | torch.float32  |     0.006 MB | L1:○  L2:○
head                   |      769,000 |   0.9% | torch.float32  |     3.076 MB | L1:✕  L2:○
cls_token              |          768 |   0.0% | torch.float32  |     0.003 MB | L1:○  L2:○
pos_embed              |      151,296 |   0.2% | torch.float32  |     0.605 MB | L1:✕  L2:○

===== ブロック1個の中の部品 =====
name                   |       params |    割合 | dtype          |           容量 | 収まるか
norm1                  |        1,536 |   0.0% | torch.float32  |     0.006 MB | L1:○  L2:○
attn.qkv               |    1,771,776 |   2.0% | torch.float32  |     7.087 MB | L1:✕  L2:✕
attn.proj           

In [25]:
# 上記を詳しく見ている
# ---- 比べる相手：L1・L2の容量 ----
cc = torch.cuda.get_device_capability(0)
L1_BYTES = {(7, 5): 64 * 1024}[cc]   # T4（Turing）：L1として使えるのは最大64KB（Tuning Guide）

_cuda = ctypes.CDLL("libcuda.so"); _cuda.cuInit(0)
_dev, _v = ctypes.c_int(), ctypes.c_int()
_cuda.cuDeviceGet(ctypes.byref(_dev), 0)
_cuda.cuDeviceGetAttribute(ctypes.byref(_v), 38, _dev)   # 38 = L2キャッシュの容量
L2_BYTES = _v.value

def fits(nbytes, cache_bytes):
    return "○" if nbytes <= cache_bytes else "✕"

# ---- 中間データの形を forward hook で取る ----
def collect_activations(bs):
    records, hooks = [], []

    def make_hook(name):
        def hook(module, inputs, output):
            out = output[0] if isinstance(output, tuple) else output
            records.append({"名前": name, "種類": type(module).__name__,
                            "出力のshape": tuple(out.shape), "要素数": out.numel()})
        return hook

    # ブロック1個目の中の一番下の部品（Linear、LayerNorm、GELUなど）
    for name, module in model.blocks[0].named_modules():
        # print(f'name: {name}')
        if name == "" or len(list(module.children())) > 0: # list(module.children())): [Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16)), Identity()]
            continue
        if isinstance(module, (nn.Identity, nn.Dropout)) or type(module).__name__ == "DropPath":
            continue
        hooks.append(module.register_forward_hook(make_hook(name))) # 関数を設定してそれをregister_forward_hookで呼び出すとprint分などが表示できる
    # ブロック1個目そのものの出力
    hooks.append(model.blocks[0].register_forward_hook(make_hook("ブロックの出力")))

    p = next(model.parameters())
    x = torch.randn(bs, 3, 224, 224, device=p.device, dtype=p.dtype)
    with torch.inference_mode():
        out = model(x)
    for h in hooks:
        h.remove()

    # attention行列は部品の出力ではないので、形から計算する（一時的な配列だから、らしい）
    n_tokens = records[0]["出力のshape"][1]
    heads = model.blocks[0].attn.num_heads
    records.append({"名前": "attention行列（計算値）", "種類": "-",
                    "出力のshape": (bs, heads, n_tokens, n_tokens),
                    "要素数": bs * heads * n_tokens * n_tokens})

    df = pd.DataFrame(records)
    df.insert(0, "bs", bs)
    for dt, b in [("FP32", 4), ("FP16", 2)]:
        df[f"{dt}(MB)"] = df["要素数"] * b / 1e6
        df[f"{dt} L1"] = (df["要素数"] * b).apply(lambda n: fits(n, L1_BYTES))
        df[f"{dt} L2"] = (df["要素数"] * b).apply(lambda n: fits(n, L2_BYTES))
    return df, out

acts_df = pd.concat([collect_activations(1)[0], collect_activations(64)[0]], ignore_index=True)
out = collect_activations(64)[1]

pd.options.display.float_format = "{:,.3f}".format
print(f"L1（最大）: {L1_BYTES/1024:.0f} KiB ／ L2: {L2_BYTES/1024**2:.1f} MiB")
display(acts_df)

# FP16の値はモデルを動かしたわけではなく理論値

L1（最大）: 64 KiB ／ L2: 4.0 MiB


,bs,名前,種類,出力のshape,要素数,FP32(MB),FP32 L1,FP32 L2,FP16(MB),FP16 L1,FP16 L2
0,1,norm1,LayerNorm,"(1, 197, 768)",151296,0.605,✕,○,0.303,✕,○
1,1,attn.qkv,Linear,"(1, 197, 2304)",453888,1.816,✕,○,0.908,✕,○
2,1,attn.proj,Linear,"(1, 197, 768)",151296,0.605,✕,○,0.303,✕,○
3,1,norm2,LayerNorm,"(1, 197, 768)",151296,0.605,✕,○,0.303,✕,○
4,1,mlp.fc1,Linear,"(1, 197, 3072)",605184,2.421,✕,○,1.210,✕,○
5,1,mlp.act,GELU,"(1, 197, 3072)",605184,2.421,✕,○,1.210,✕,○
6,1,mlp.fc2,Linear,"(1, 197, 768)",151296,0.605,✕,○,0.303,✕,○
7,1,ブロックの出力,Block,"(1, 197, 768)",151296,0.605,✕,○,0.303,✕,○
8,1,attention行列（計算値）,-,"(1, 12, 197, 197)",465708,1.863,✕,○,0.931,✕,○
9,64,norm1,LayerNorm,"(64, 197, 768)",9682944,38.732,✕,✕,19.366,✕,✕


In [16]:
# フック関数の定義について確認
# 旧シグネチャ: hook(module, input, output)
# 新しい推奨シグネチャ: hook(module, args, kwargs, output) ※バージョンにより異なります
def forward_hook_fn(module, input, output):
  print(f"Module: {module.__class__.__name__}")
  print(f"Input shape: {input[0].shape}")
  print(f"Output shape: {output.shape}")
  # 必要に応じてoutputを書き換えて返すことも可能


# 例：モデルの特定レイヤーに登録
model = nn.Linear(10, 5)
handle = model.register_forward_hook(forward_hook_fn)

In [17]:
x = torch.randn(1, 10)
y = model(x)

Module: Linear
Input shape: torch.Size([1, 10])
Output shape: torch.Size([1, 5])


In [30]:
# 入力テンソルの容量
bs=64
x = torch.randn(bs, 3, 224, 224, device=p.device, dtype=p.dtype)
nbytes = x.numel() * x.element_size()
print(f"{nbytes:,} バイト = {nbytes / 1e6:.3f} MB")

# 出力テンソルの容量
nbytes = out.numel() * out.element_size()
print(f"{nbytes:,} バイト = {nbytes / 1e6:.3f} MB")

# モデル: 346.3 MB

38,535,168 バイト = 38.535 MB
256,000 バイト = 0.256 MB


In [31]:
# カタログの理論値: 320GB/s
# 下限 ＝ 重みやI/Oテンソル÷ メモリバンド幅
(346.3 + 38.535 + 0.256)/320

1.2034093750000001